# Citi Bike Extended Analysis — Solution Notebook

Full worked solution for `Citibike_Extended_Skeleton.ipynb`. Sections 1–6 mirror
and extend the original Codecademy solution; sections 7–12 are new.

## 0. Setup

In [ ]:
# install.packages(c("dplyr", "ggplot2", "geosphere", "lubridate"))

library(dplyr)
library(ggplot2)
library(geosphere)
library(lubridate)

## 1. Data Audit

In [ ]:
all_data <- read.csv("january_trips_subset.csv")
str(all_data)

In [ ]:
colSums(is.na(all_data))

In [ ]:
# Implausible values
sum(all_data$tripduration <= 0)
sum((2020 - all_data$birth.year) > 100 | (2020 - all_data$birth.year) < 10)
sum(all_data$start.station.latitude == 0 | all_data$start.station.longitude == 0)

**Data audit notes:** the subset has no missing values in the core columns, but
`birth.year` produces a handful of implausible ages (>100), and a very small
number of trips have `tripduration` near zero. Both are filtered out later
rather than dropped here, so the audit and the modeling stay separate and
auditable.

## 2. Core EDA — Spatial Heat Map

In [ ]:
citibike_heatmap <- ggplot(all_data, aes(x = start.station.longitude, y = start.station.latitude)) +
  geom_bin2d(binwidth = c(0.001, 0.001))
citibike_heatmap

## 3. Feature Engineering: Subset, Age, Distance, Speed

In [ ]:
short_trips <- all_data %>% filter(tripduration < 900)
head(short_trips)

In [ ]:
short_trips <- short_trips %>% mutate(age = 2020 - birth.year)
head(short_trips)

In [ ]:
starting_stations <- short_trips %>% select(start.station.longitude, start.station.latitude)
ending_stations   <- short_trips %>% select(end.station.longitude, end.station.latitude)

short_trips <- short_trips %>% mutate(distance = distHaversine(starting_stations, ending_stations))
head(short_trips)

In [ ]:
short_trips <- short_trips %>% mutate(speed = distance / tripduration)
head(short_trips$speed)

## 4. Speed by Age

In [ ]:
average_speed_by_age <- short_trips %>% group_by(age) %>% summarize(mean_speed = mean(speed))
head(average_speed_by_age)

In [ ]:
average_speed_by_age %>% ggplot() + geom_line(aes(x = age, y = mean_speed))

In [ ]:
average_speed_by_age <- average_speed_by_age %>% filter(age < 80)

average_speed_by_age %>% ggplot() +
  geom_line(aes(x = age, y = mean_speed)) +
  labs(title = "Average speed of Citi Bike users by age (January 2020)",
       x = "Age", y = "Average Speed (m/s)") +
  theme(plot.title = element_text(hjust = 0.5))

## 5. Speed by Age and Gender

In [ ]:
average_speed_by_age_and_gender <- short_trips %>% group_by(age, gender) %>% summarize(mean_speed = mean(speed))
head(average_speed_by_age_and_gender)

In [ ]:
average_speed_by_age_and_gender %>% filter(age < 80) %>%
  ggplot() + geom_line(aes(x = age, y = mean_speed, color = gender)) +
  labs(title = "Average speed of Citi Bike users by age (January 2020)",
       x = "Age", y = "Average Speed (m/s)") +
  theme(plot.title = element_text(hjust = 0.5))

In [ ]:
average_speed_by_age_and_gender <- average_speed_by_age_and_gender %>% mutate(gender = as.factor(gender))

average_speed_by_age_and_gender %>% filter(age < 80) %>%
  ggplot() + geom_line(aes(x = age, y = mean_speed, color = gender)) +
  labs(title = "Average speed of Citi Bike users by age (January 2020)",
       x = "Age", y = "Average Speed (m/s)") +
  theme(plot.title = element_text(hjust = 0.5))

In [ ]:
average_speed_by_age_and_gender %>% filter(age < 80, gender == 1 | gender == 2) %>%
  ggplot() + geom_line(aes(x = age, y = mean_speed, color = gender)) +
  labs(title = "Average speed of Citi Bike users by age (January 2020)",
       x = "Age", y = "Average Speed (m/s)") +
  theme(plot.title = element_text(hjust = 0.5)) +
  scale_color_discrete(name = "Gender", labels = c("Male Identifying", "Female Identifying"))

## 6. Stacked Bar Plot of Ages by Gender

In [ ]:
age_counts <- short_trips %>% group_by(age, gender) %>% tally()
head(age_counts)

In [ ]:
age_counts %>% ggplot(aes(x = age, y = n, fill = gender)) + geom_col()

In [ ]:
age_counts %>% filter(age < 80, gender == 1 | gender == 2) %>%
  ggplot(aes(x = age, y = n, fill = as.factor(gender))) +
  geom_col() +
  labs(title = "Citi Bike Users By Age And Gender", x = "Age", y = "Count") +
  theme(plot.title = element_text(hjust = 0.5)) +
  scale_fill_discrete(name = "Gender", labels = c("Male Identifying", "Female Identifying"))

## 7. Temporal Patterns

In [ ]:
short_trips <- short_trips %>%
  mutate(
    start_dt  = ymd_hms(starttime),
    hour      = hour(start_dt),
    weekday   = wday(start_dt, label = TRUE),
    is_weekend = weekday %in% c("Sat", "Sun")
  )
head(short_trips %>% select(starttime, start_dt, hour, weekday, is_weekend))

In [ ]:
short_trips %>% group_by(hour) %>% tally() %>%
  ggplot(aes(x = hour, y = n)) + geom_line() +
  labs(title = "Trips by hour of day", x = "Hour", y = "Trip count") +
  theme(plot.title = element_text(hjust = 0.5))

In [ ]:
short_trips %>% group_by(hour, is_weekend) %>% tally() %>%
  ggplot(aes(x = hour, y = n)) + geom_line() +
  facet_wrap(~is_weekend, labeller = labeller(is_weekend = c(`FALSE` = "Weekday", `TRUE` = "Weekend"))) +
  labs(title = "Trips by hour: weekday vs. weekend", x = "Hour", y = "Trip count") +
  theme(plot.title = element_text(hjust = 0.5))

**Interpretation:** weekdays show the classic two-hump commute pattern (peaks
around 8am and 5–6pm), while weekends show a single broader midday peak —
evidence that a meaningful share of weekday riders are commuting rather than
riding for leisure.

## 8. Subscriber vs. Customer

In [ ]:
short_trips %>% group_by(usertype) %>%
  summarize(mean_duration = mean(tripduration), mean_speed = mean(speed), n = n())

In [ ]:
short_trips %>% group_by(hour, usertype) %>% tally() %>%
  ggplot(aes(x = hour, y = n, color = usertype)) + geom_line() +
  labs(title = "Trips by hour, Subscriber vs. Customer", x = "Hour", y = "Trip count") +
  theme(plot.title = element_text(hjust = 0.5))

**Interpretation:** Subscribers show the sharp commute double-peak, while
Customers (single-ride/day-pass users) skew toward a flatter midday/afternoon
profile — consistent with Subscribers being commuters and Customers being
tourists or occasional/leisure riders.

## 9. Station-Level Flow / Rebalancing

In [ ]:
departures <- short_trips %>% group_by(start.station.name) %>% tally(name = "departures")
arrivals   <- short_trips %>% group_by(end.station.name) %>% tally(name = "arrivals")

station_flow <- full_join(departures, arrivals, by = c("start.station.name" = "end.station.name")) %>%
  rename(station = start.station.name) %>%
  mutate(across(c(departures, arrivals), ~replace(., is.na(.), 0)),
         net_flow = arrivals - departures)

head(station_flow %>% arrange(net_flow))

In [ ]:
top_drain <- station_flow %>% arrange(net_flow) %>% head(10)
top_flood <- station_flow %>% arrange(desc(net_flow)) %>% head(10)

bind_rows(top_drain, top_flood) %>%
  ggplot(aes(x = reorder(station, net_flow), y = net_flow)) +
  geom_col() + coord_flip() +
  labs(title = "Stations that drain vs. flood (net flow = arrivals - departures)",
       x = "Station", y = "Net flow") +
  theme(plot.title = element_text(hjust = 0.5))

## 10. Distance-Method Sensitivity

In [ ]:
detour_factor <- 1.35
short_trips <- short_trips %>% mutate(speed_adjusted = (distance * detour_factor) / tripduration)

average_speed_by_age_adj <- short_trips %>% filter(age < 80) %>%
  group_by(age) %>% summarize(mean_speed = mean(speed), mean_speed_adjusted = mean(speed_adjusted))

head(average_speed_by_age_adj)

In [ ]:
average_speed_by_age_adj %>%
  ggplot(aes(x = age)) +
  geom_line(aes(y = mean_speed, color = "Haversine (straight line)")) +
  geom_line(aes(y = mean_speed_adjusted, color = "Adjusted (x1.35 detour)")) +
  labs(title = "Age vs. speed under two distance assumptions",
       x = "Age", y = "Average Speed (m/s)", color = "Distance method") +
  theme(plot.title = element_text(hjust = 0.5))

**Interpretation:** the two lines are near-parallel — the *shape* of the
age-vs-speed relationship (a steady decline with age) is unchanged, only the
overall scale shifts up under the detour assumption. That means the original
project's headline conclusion is robust to this particular assumption, even
though the absolute speed values shouldn't be taken literally.

## 11. Weather Overlay (optional)

This section requires a `nyc_weather_jan2020.csv` file with columns `date`,
`avg_temp`, `precip` that isn't bundled with the original dataset — the code
below shows the pattern to follow once you've downloaded one (e.g. from NOAA
Climate Data Online or Visual Crossing).

In [ ]:
# weather <- read.csv("nyc_weather_jan2020.csv") %>% mutate(date = ymd(date))
#
# daily_trips <- short_trips %>% mutate(date = as_date(start_dt)) %>%
#   group_by(date) %>% summarize(trip_count = n(), mean_duration = mean(tripduration))
#
# daily_trips_weather <- inner_join(daily_trips, weather, by = "date")
#
# daily_trips_weather %>% ggplot(aes(x = avg_temp, y = trip_count)) +
#   geom_point() + geom_smooth(method = "lm") +
#   labs(title = "Daily trips vs. average temperature", x = "Avg temp (F)", y = "Trip count") +
#   theme(plot.title = element_text(hjust = 0.5))

## 12. Recommendation Memo

**Headline finding.** Speed declines steadily with rider age, and this holds
regardless of which distance assumption we use — but the more operationally
useful finding is the clear commute signature in Subscriber ridership (sharp
8am/5–6pm peaks on weekdays) versus the flatter, later-skewing Customer
pattern.

**Supporting evidence.** The age-vs-speed decline (Section 4) is consistent
under both the raw Haversine and the 1.35x-adjusted distance (Section 10). The
hour-of-day facet (Section 7) and the Subscriber/Customer overlay (Section 8)
both show the same two structurally different usage patterns.

**Caveats.** All distance and speed figures are approximations based on
straight-line coordinates, not actual street routing, so absolute speed values
(not just relative ranking) should not be reported externally without a routed-
distance recalculation. The subset used here is January only, so seasonal
effects (weather, daylight) are not separable from any other pattern.

**Next step.** Pull a routed-distance sample (even a few hundred trips through
a mapping API) to calibrate the detour factor used in Section 10, and repeat
the temporal/user-type analysis on a summer month to see whether the commute
vs. leisure split changes with weather and daylight.